# `StructurePreprocessor.fetch_alphafold()`

`fetch_alphafold` bulk-downloads each entry's AlphaFold-DB model file **and** its Predicted Aligned Error (PAE) sidecar from https://alphafold.ebi.ac.uk into one folder, saving them under the names `encode_pdb` / `encode_pae` / `get_dssp` already resolve, so a single call populates the folder the encoders consume. It is the `fetch_` (web) acquisition verb, the structure-side analog of `AnnotationPreprocessor.fetch_uniprot`.

It returns a per-entry status DataFrame (`entry`, `model_ok`, `pae_ok`, `alphafold_ok`, `skipped`, `model_path`, `pae_path`). A 404 (accession not in AlphaFold DB, or a fragmented `F2+` protein) is the soft failure governed by `on_failure`; other network errors raise `RuntimeError`.

Requires `aaanalysis[pro]` and network access.

In [1]:
import warnings
import tempfile
from pathlib import Path
import aaanalysis as aa
aa.options['verbose'] = False
warnings.filterwarnings('ignore')

# Three short human proteins from the bundled gamma-secretase set; their ``entry``
# values are UniProt accessions, which is what AlphaFold DB is keyed by.
df_seq = aa.load_dataset(name='DOM_GSEC', n=10)
df_seq = df_seq[df_seq['entry'].isin(['Q14802', 'O43914', 'P01135'])].reset_index(drop=True)

strp = aa.StructurePreprocessor(verbose=False)
af_dir = Path(tempfile.mkdtemp()) / 'alphafold'   # use a persistent project folder in real work

df_status = strp.fetch_alphafold(df_seq=df_seq, out_folder=af_dir)
aa.display_df(df_status[['entry', 'model_ok', 'pae_ok', 'alphafold_ok', 'skipped']], n_rows=10, show_shape=True)

DataFrame shape: (3, 5)


,entry,model_ok,pae_ok,alphafold_ok,skipped
1,Q14802,True,True,True,False
2,P01135,True,True,True,False
3,O43914,True,True,True,False


The folder now holds one `<entry>.pdb` model and one `AF-<entry>-F1-predicted_aligned_error_v4.json` sidecar per entry, exactly where the encoders look for them, so you can encode straight from it:

In [2]:
print(sorted(p.name for p in af_dir.iterdir()))

dict_pdb = strp.encode_pdb(df_seq=df_seq, pdb_folder=af_dir, features=['plddt'])
dict_pae = strp.encode_pae(df_seq=df_seq, pae_folder=af_dir, features=['pae_row_mean'])
print({entry: (dict_pdb[entry].shape, dict_pae[entry].shape) for entry in dict_pdb})

['AF-O43914-F1-predicted_aligned_error_v4.json', 'AF-P01135-F1-predicted_aligned_error_v4.json', 'AF-Q14802-F1-predicted_aligned_error_v4.json', 'O43914.pdb', 'P01135.pdb', 'Q14802.pdb']
{'Q14802': ((87, 1), (87, 1)), 'P01135': ((160, 1), (160, 1)), 'O43914': ((113, 1), (113, 1))}


## Further parameters

`file_format` selects the structure file type (`'pdb'` or the modern `'cif'`); `timeout` bounds each request in seconds; `skip_existing=True` leaves files that are already present alone and re-fetches only what is missing (here the PAE sidecars are reused and only the `.cif` models are downloaded); `on_failure` governs 404 soft failures (`'nan'` keeps the row marked not-ok, `'drop'` removes it, `'raise'` raises); `return_df=True` also returns an echo of `df_seq` with an `alphafold_ok` column; `max_workers > 1` downloads on a thread pool with input-ordered, byte-identical results (opt-in, because parallel AlphaFold-DB requests risk HTTP-429 throttling).

In [3]:
df_status_cif, df_seq_out = strp.fetch_alphafold(df_seq=df_seq, out_folder=af_dir, file_format='cif',
                                                 timeout=60.0, skip_existing=True, on_failure='nan',
                                                 return_df=True, max_workers=2)
aa.display_df(df_seq_out[['entry', 'gene', 'alphafold_ok']], n_rows=10, show_shape=True)

DataFrame shape: (3, 3)


,entry,gene,alphafold_ok
1,Q14802,FXYD3,True
2,P01135,TGFA,True
3,O43914,TYROBP,True
